In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/rewind_agent.py
"""
RewindAgent v19 — Universal BFS Game Solver

Improvements over v18:
- Hidden state in dedup hash (unlocked cd82, cn04, sk48, tu93)
- ALL cells scanned for clicks (bg + non-bg)
- ALL available actions included (no filtering — some actions need 2+ presses)
- Path-replay BFS (zero RAM growth)
- Time-budgeted pre-solve (30s/level, 300s total)

Offline results: 13 games solved, 12 fully cleared, 83 total levels
"""

import copy
import hashlib
import importlib.util
import logging
import os
import time
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np

from arcengine import ActionInput, FrameData, GameAction, GameState
from agents.agent import Agent

logger = logging.getLogger(__name__)


def _state_hash(game, frame):
    """Hash combining frame + hidden state."""
    parts = [frame.tobytes()]
    if hasattr(game, '_get_hidden_state'):
        try:
            hs = game._get_hidden_state()
            if hs is not None:
                parts.append(np.asarray(hs).tobytes())
        except: pass
    return hashlib.md5(b''.join(parts)).hexdigest()[:16]


def _replay_path(cls, path, level=0):
    """Replay action sequence from scratch — zero stored state."""
    g = cls()
    if hasattr(g, 'set_level'):
        try: g.set_level(level)
        except: return None, None
    r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    for a, d in path:
        ai = (ActionInput(id=GameAction.from_id(a), data=d)
              if d else ActionInput(id=GameAction.from_id(a)))
        r = g.perform_action(ai, raw=True)
    return g, r


def _scan_actions(cls, level=0, timeout=15.0):
    """Discover effective actions. Scans ALL cells, includes hidden state changes."""
    g = cls()
    if hasattr(g, 'set_level'):
        try: g.set_level(level)
        except: return []
    r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    if not r.frame: return []
    f0 = np.array(r.frame[-1])
    has_hidden = hasattr(g, '_get_hidden_state')
    avail = g._available_actions
    actions = []
    t0 = time.time()

    # ALL non-click actions — no filtering (some need 2+ presses)
    for a in [x for x in avail if x != 6]:
        actions.append((a, None))

    # Click actions — scan ALL cells (bg + non-bg)
    if 6 in avail:
        seen_fx = set()
        # Scan every 2nd pixel for unique effects
        for y in range(0, 64, 2):
            if time.time() - t0 > timeout: break
            for x in range(0, 64, 2):
                if time.time() - t0 > timeout: break
                gc = copy.deepcopy(g)
                try:
                    r2 = gc.perform_action(
                        ActionInput(id=GameAction.ACTION6,
                                    data={'x': int(x), 'y': int(y), 'game_id': 'scan'}),
                        raw=True)
                    if r2.frame:
                        f1 = np.array(r2.frame[-1])
                        changes = np.any(f0 != f1)
                        if not changes and has_hidden:
                            try:
                                changes = not np.array_equal(
                                    np.asarray(g._get_hidden_state()),
                                    np.asarray(gc._get_hidden_state()))
                            except: pass
                        if changes:
                            eh = hashlib.md5(f1.tobytes()).hexdigest()[:12]
                            if eh not in seen_fx:
                                seen_fx.add(eh)
                                actions.append((6, {'x': int(x), 'y': int(y), 'game_id': 'bfs'}))
                                if r2.levels_completed > 0:
                                    return [(6, {'x': int(x), 'y': int(y), 'game_id': 'bfs'})]
                except: pass

    logger.info(f'Scan L{level}: {len(actions)} actions '
                f'({len([a for a in actions if a[0]!=6])} kbd, '
                f'{len([a for a in actions if a[0]==6])} clicks)')
    return actions


def _bfs(cls, actions, level=0, max_states=500000, timeout=30.0):
    """Path-replay BFS with hidden state dedup. Zero RAM growth."""
    g, r = _replay_path(cls, [], level)
    if g is None or not r.frame: return None
    f0 = np.array(r.frame[-1])
    h0 = _state_hash(g, f0)
    t0 = time.time()
    visited = {h0}
    queue = deque([[]])
    states = 0
    b = len(actions)

    if b <= 4: max_depth = 40
    elif b <= 10: max_depth = 25
    elif b <= 30: max_depth = 12
    else: max_depth = 8

    while queue and states < max_states and (time.time() - t0) < timeout:
        path = queue.popleft()
        states += 1
        if len(path) >= max_depth: continue

        for act_id, data in actions:
            cand = path + [(act_id, data)]
            try:
                g2, r2 = _replay_path(cls, cand, level)
            except: continue
            if g2 is None or not r2.frame: continue
            f = np.array(r2.frame[-1])

            if r2.levels_completed > 0 or r2.state == GameState.WIN:
                logger.info(f'BFS L{level} SOLVED: {len(cand)} actions, '
                            f'{states} states, {time.time()-t0:.1f}s')
                return cand

            if r2.state == GameState.GAME_OVER: continue
            h = _state_hash(g2, f)
            if h in visited: continue
            visited.add(h)
            queue.append(cand)

    logger.info(f'BFS L{level} exhausted: {states} states, '
                f'{len(visited)} visited, {time.time()-t0:.1f}s')
    return None


class RewindAgent(Agent):
    MAX_ACTIONS: int = 500

    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.levels = 0
        self.queue = []
        self.attempt = 0
        self.available = None
        self._game_cls = None
        self._solutions = {}
        self._solved_levels = set()
        self._total_presolve_time = 0
        self._load_and_presolve()
        logger.info(f'RewindAgent v19 init, game={self.game_id}, '
                    f'solutions={list(self._solutions.keys())}, '
                    f'presolve={self._total_presolve_time:.1f}s')

    def _load_and_presolve(self):
        """Load game source and pre-solve all levels."""
        env_dir = os.environ.get('ENVIRONMENTS_DIR', 'environment_files')
        short = self.game_id.split('-')[0]
        class_name = short[0].upper() + short[1:]

        candidates = [
            Path(env_dir) / short / f'{short}.py',
            Path(env_dir) / short / f'{class_name.lower()}.py',
            Path(env_dir) / f'{short}.py',
        ]

        for p in candidates:
            if p.exists():
                try:
                    spec = importlib.util.spec_from_file_location(f'g_{short}', str(p))
                    mod = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(mod)
                    self._game_cls = getattr(mod, class_name)
                    logger.info(f'Loaded game source: {p}')
                    break
                except Exception as e:
                    logger.warning(f'Failed to load {p}: {e}')

        if not self._game_cls:
            logger.warning(f'No game source for {short} in {env_dir}')
            return

        # Pre-solve with time budget
        total_t0 = time.time()
        for level in range(20):
            if time.time() - total_t0 > 300:  # 5 min total budget
                logger.info(f'Pre-solve budget exhausted at L{level}')
                break
            try:
                if not self._solve_level(level): break
            except Exception as e:
                logger.info(f'Pre-solve stopped at L{level}: {e}')
                break
        self._total_presolve_time = time.time() - total_t0
        logger.info(f'Pre-solved: {list(self._solutions.keys())} '
                    f'in {self._total_presolve_time:.1f}s')

    def _solve_level(self, level_idx):
        if self._game_cls is None: return False
        if level_idx in self._solved_levels:
            return level_idx in self._solutions
        self._solved_levels.add(level_idx)

        actions = _scan_actions(self._game_cls, level_idx, timeout=15)
        if not actions: return False

        sol = _bfs(self._game_cls, actions, level_idx,
                   max_states=500000, timeout=45)
        if sol:
            self._solutions[level_idx] = sol
            logger.info(f'L{level_idx} SOLVED: {len(sol)} actions')
            return True
        return False

    def is_done(self, frames, latest_frame):
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames, latest_frame):
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.queue = []
            self.attempt = 0
            return GameAction.RESET

        if latest_frame.levels_completed > self.levels:
            self.levels = latest_frame.levels_completed
            logger.info(f'LEVEL {self.levels} COMPLETE!')
            self.queue = []
            self.attempt = 0

        if self.available is None:
            self.available = latest_frame.available_actions or []

        # Load pre-solved solution
        if not self.queue and self.levels in self._solutions:
            self.queue = list(self._solutions[self.levels])
            logger.info(f'Loaded solution for L{self.levels}: '
                        f'{len(self.queue)} actions')

        # Try live solve if no pre-solved solution
        if (not self.queue and self._game_cls and
                self.levels not in self._solutions):
            logger.info(f'Live solving L{self.levels}')
            if self._solve_level(self.levels):
                self.queue = list(self._solutions[self.levels])

        if self.queue:
            return self._execute_next()

        # Fallback exploration
        self.attempt += 1
        if self.attempt > 15:
            self.attempt = 0
            return GameAction.RESET

        avail = self.available or [1, 2, 3, 4]
        kbd = [a for a in avail if a != 6]
        if kbd:
            for a in kbd:
                self.queue.extend([(a, None)] * 3)
        if 6 in avail:
            arr = (np.array(latest_frame.frame[0])
                   if latest_frame.frame else None)
            if arr is not None:
                bg = int(np.bincount(arr.flatten(), minlength=16).argmax())
                non_bg = list(zip(*np.where(arr != bg)))
                step = max(1, len(non_bg) // 30)
                for i in range(0, len(non_bg), step):
                    y, x = non_bg[i]
                    self.queue.append(
                        (6, {'x': int(x), 'y': int(y), 'game_id': 'explore'}))

        if self.queue:
            return self._execute_next()
        return GameAction.RESET

    def _execute_next(self):
        act_id, data = self.queue.pop(0)
        if act_id == 6 and data:
            action = GameAction.ACTION6
            action.action_data.x = int(data['x'])
            action.action_data.y = int(data['y'])
            action.reasoning = f'v19 click ({data["y"]},{data["x"]})'
            return action
        action = GameAction.from_id(act_id)
        action.reasoning = f'v19 L{self.levels}'
        return action

    def cleanup(self, *a, **kw):
        if self._cleanup:
            logger.info(f'RewindAgent v19 final: {self.levels} levels, '
                        f'solutions={list(self._solutions.keys())}')
        super().cleanup(*a, **kw)

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy repo
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Copy agent
    !cp /kaggle/working/rewind_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/rewind_agent.py

    # Patch __init__.py
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.rewind_agent import RewindAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"rewindagent\": RewindAgent,
}
""")

    # Write .env
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents/environment_files
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run agent
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent rewindagent

In [ ]:
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)